# RSNA Knee — v20b oofmask SWA ep8/9/11, **solo**

Notebook **self-contained** pour Kaggle (internet off, **T4**), **un seul modèle** :
DICOM test → crop 130 mm + bande 20–80 % → volumes (24, 224, 224)
→ **KneeNet v20b** (`v20b_crop130_oofmask_full`, SWA uniforme ep8+ep9+ep11)
→ TTA affine ×5 → `submission.csv`.

**SWA** : moyenne uniforme des poids ep8+ep9+ep11 (BN gelée, pas de re-estimation).
Un seul modèle, pas un ensemble de prédictions.

Ce run a ete train **sans** canon laterality L/R : **pas de flip**.
Ne pas mixer avec un checkpoint v20h flipped.

Poids (dataset `sultent/rsna-knee-v20b-swa`) :
- `v20b_swa_ep8_9_11.pt` — optimizer retire, 6 slots FS/T1, ResNet-50 RadImageNet

TTA : identite + rotation ±5° + scale 0.95/1.05. **Pas de hflip**.
Le kernel **doit** ecrire `submission.csv` (cwd / `/kaggle/working`).


In [ ]:
import os
from pathlib import Path

KAGGLE = os.path.exists('/kaggle')
WEIGHT_NAME = 'v20b_swa_ep8_9_11.pt'
if KAGGLE:
    DATA_DIR = Path('/kaggle/input')
    print('contenu /kaggle/input :', sorted(p.name for p in DATA_DIR.iterdir()))
    COMP_DIR = None
    for base in [DATA_DIR / 'competitions', DATA_DIR]:
        if base.exists():
            for p in sorted(base.iterdir()):
                if (p / 'test.csv').exists():
                    COMP_DIR = p
                    break
        if COMP_DIR is not None:
            break
    if COMP_DIR is None:
        raise FileNotFoundError('test.csv introuvable sous /kaggle/input')
    search_roots = []
    if (DATA_DIR / 'datasets').exists():
        search_roots.append(DATA_DIR / 'datasets')
    search_roots.append(DATA_DIR)

    def _find_weight(name):
        for root in search_roots:
            hits = list(root.rglob(name))
            if hits:
                return hits[0]
        raise FileNotFoundError(f'{name} introuvable (dataset v20b swa attache ?)')

    WEIGHTS_PATH = _find_weight(WEIGHT_NAME)
else:
    COMP_DIR = Path('../data/csv')
    WEIGHTS_PATH = Path('../kaggle_dataset_v20b/_upload_swa') / WEIGHT_NAME

TEST_CSV = COMP_DIR / 'test.csv'
TEST_SERIES_CSV = COMP_DIR / 'test_series.csv'
TEST_SERIES_DIR = COMP_DIR / 'test_series'
SAMPLE_SUB = COMP_DIR / 'sample_submission.csv'

import torch

def _pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
        _ = m(torch.zeros(1, 3, 32, 32, device='cuda')).sum().item()
        return 'cuda'
    except Exception as e:
        print('CUDA inutilisable, repli CPU :', str(e)[:120])
        return 'cpu'

DEVICE = _pick_device()
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

AMP_DTYPE = (torch.bfloat16
             if (DEVICE == 'cuda' and torch.cuda.is_bf16_supported())
             else torch.float16)
DEBUG_COMPARE_NPZ = True
NPZ_MEAN_U8, NPZ_STD_U8 = 42.0, 49.5

TTA_VARIANTS = (
    (0.0, 1.00),
    (5.0, 1.00),
    (-5.0, 1.00),
    (0.0, 0.95),
    (0.0, 1.05),
)
# v20b oofmask full : PAS de laterality. Ne pas flipper.
LATERALITY_CANON_L = False

print('competition dir :', COMP_DIR)
print('poids           :', WEIGHTS_PATH, '| existe :', WEIGHTS_PATH.exists())
print('device          :', DEVICE)
print('TTA variantes   :', TTA_VARIANTS)
print('laterality      :', LATERALITY_CANON_L, '(v20b non-flipped)')
print('modele          : v20b oofmask SWA ep8/9/11')


In [ ]:
# --- Pretraitement DICOM -> volume (24, 224, 224) uint8 --------------------
# (copie self-contained de src/infer/preprocess_dicom.py)
# crop 130 mm + bande 20-80 % (air / bords), identique a la conversion NPZ
import cv2
import numpy as np
import pydicom

N_SLICES, IMG_SIZE = 24, 224
CROP_MM, CROP_REL, BAND = 130.0, 0.70, (0.20, 0.80)


def _slice_sort_key(ds):
    inst = getattr(ds, 'InstanceNumber', None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, 'ImagePositionPatient', None)
    iop = getattr(ds, 'ImageOrientationPatient', None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    return float(ipp[2]) if ipp is not None else 0.0


def _spacing_mm(ds):
    ps = getattr(ds, 'PixelSpacing', None) or getattr(ds, 'ImagerPixelSpacing', None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img, row_mm, col_mm):
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = min(h, max(8, int(round(CROP_MM / row_mm))))
        cw = min(w, max(8, int(round(CROP_MM / col_mm))))
    else:
        ch = min(h, max(8, int(round(h * CROP_REL))))
        cw = min(w, max(8, int(round(w * CROP_REL))))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]


_SERIES_CACHE = {}

def load_series_dicom(series_dir):
    key = str(Path(series_dir))
    if key in _SERIES_CACHE:
        return _SERIES_CACHE[key]
    files = sorted(Path(series_dir).glob('*.dcm'))
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError(f'aucun DICOM lisible dans {series_dir}')
    datasets.sort(key=_slice_sort_key)

    frames, sp = [], None
    for ds in datasets:
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * float(getattr(ds, 'RescaleSlope', 1.0) or 1.0) \
                  + float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
        if getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2') == 'MONOCHROME1':
            arr = arr.max() - arr
        if sp is None:
            sp = _spacing_mm(ds)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f'ndim inattendu {arr.ndim}')
    if not frames:
        raise RuntimeError('aucune coupe')

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped = [_center_crop(im, row_mm, col_mm) for im in frames]
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)
    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]
    out = np.empty((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for i in range(N_SLICES):
        sl = cv2.resize(band[i], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)
    _SERIES_CACHE[key] = out
    return out


PROTOCOL_SLOTS = [
    ('Sagittal', 1),
    ('Coronal', 1),
    ('Axial', 1),
    ('Sagittal', 0),
    ('Coronal', 0),
    ('Axial', 0),
]


def _pick_protocol_row(series_rows, plane, fluid):
    '''Un contraste exact (FS ou T1) sur un plan. Pas de repli.'''
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    if 'Fluid_Sensitive' in rows.columns:
        rows = rows[rows.Fluid_Sensitive == int(fluid)]
    elif fluid == 1 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 1]
    elif fluid == 0 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 0]
    if len(rows) == 0:
        return None
    return rows.sort_values('SeriesInstanceUID').iloc[0]


LAT_MIN_OFFSET_MM = 20.0


def _lat_side_tag(v):
    if v is None:
        return None
    s = str(v).strip().upper()
    return s[0] if s and s[0] in ('L', 'R') else None


def _lat_center_x(ipp, iop, ps, rows, cols):
    try:
        ipp = np.asarray(list(ipp), dtype=np.float64)
        iop = np.asarray(list(iop), dtype=np.float64)
        ps = np.asarray(list(ps), dtype=np.float64)
        rows, cols = float(rows), float(cols)
    except (TypeError, ValueError):
        return None
    if ipp.size < 3 or iop.size < 6 or ps.size < 2:
        return None
    c = ipp[:3] + iop[:3] * ps[1] * cols / 2.0 + iop[3:6] * ps[0] * rows / 2.0
    x = float(c[0])
    return x if np.isfinite(x) else None


def laterality_from_study_dir(study_dir):
    '''Canon L : tag Laterality, sinon X centre (zone morte 20 mm).'''
    study_dir = Path(study_dir)
    xs, tags = [], []
    if not study_dir.is_dir():
        return None
    for series in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        files = sorted(series.glob('*.dcm'))
        if not files:
            continue
        try:
            ds = pydicom.dcmread(str(files[len(files) // 2]),
                                 stop_before_pixels=True, force=True)
        except Exception:
            continue
        t = _lat_side_tag(getattr(ds, 'Laterality', None))
        if t:
            tags.append(t)
        x = _lat_center_x(
            getattr(ds, 'ImagePositionPatient', None),
            getattr(ds, 'ImageOrientationPatient', None),
            getattr(ds, 'PixelSpacing', None),
            getattr(ds, 'Rows', None), getattr(ds, 'Columns', None))
        if x is not None:
            xs.append(x)
    if tags:
        return tags[0]
    if not xs:
        return None
    m = float(np.median(xs))
    if abs(m) < LAT_MIN_OFFSET_MM:
        return None
    return 'R' if m < 0 else 'L'


def normalise_laterality(vol, plane, lat):
    if lat != 'R':
        return vol
    plane_c = str(plane or '').strip().capitalize()
    if plane_c in ('Coronal', 'Axial'):
        return np.ascontiguousarray(np.flip(vol, axis=-1))
    if plane_c == 'Sagittal':
        return np.ascontiguousarray(np.flip(vol, axis=0))
    return vol


def study_to_tensor(study_dir, series_rows, n_series=3, laterality=False):
    '''(n_series, 24, 224, 224) ; n_series=6 : 3 FS + 3 T1, sans repli.'''
    planes = ['Sagittal', 'Coronal', 'Axial']
    out = np.zeros((n_series, N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    used = set()
    lat = laterality_from_study_dir(study_dir) if laterality else None

    def _fill(i, pick, plane):
        if pick is None:
            return
        sdir = Path(study_dir) / pick.SeriesInstanceUID
        if not sdir.exists():
            return
        try:
            vol = load_series_dicom(sdir).astype(np.float32) / 255.0
            if laterality:
                vol = normalise_laterality(vol, plane, lat)
            out[i] = vol
            used.add(str(pick.SeriesInstanceUID))
        except Exception as e:
            print(f'  [warn] {pick.SeriesInstanceUID}: {e}')

    if n_series == 6:
        for i, (plane, fluid) in enumerate(PROTOCOL_SLOTS):
            _fill(i, _pick_protocol_row(series_rows, plane, fluid), plane)
        return out

    for i, plane in enumerate(planes):
        rows = series_rows[series_rows.Anatomical_Plane == plane]
        if len(rows) == 0:
            continue
        fluid = rows[rows.Fluid_Sensitive == 1] if 'Fluid_Sensitive' in rows else rows.iloc[0:0]
        if len(fluid) == 0 and 'Fat_Suppression' in rows:
            fluid = rows[rows.Fat_Suppression == 1]
        pick = (fluid if len(fluid) else rows).sort_values('SeriesInstanceUID').iloc[0]
        _fill(i, pick, plane)
    if n_series >= 4:
        t1 = None
        for plane in planes:
            rows = series_rows[series_rows.Anatomical_Plane == plane]
            if 'Fluid_Sensitive' in rows:
                rows = rows[rows.Fluid_Sensitive == 0]
            rows = rows[~rows.SeriesInstanceUID.isin(used)]
            if len(rows):
                t1 = rows.sort_values('SeriesInstanceUID').iloc[0]
                break
        pl = None if t1 is None else str(t1.Anatomical_Plane)
        _fill(3, t1, pl)
    return out


In [ ]:
# --- KneeNet v20b (copie self-contained de src/models/kneenet_v20b.py) ---
# -*- coding: utf-8 -*-
"""KneeNet v20b — v17 + 6 slots protocole (FS / T1 par plan).

Base v17 inchangee : ResNet-50 RadImageNet, stem A+B, CAM 14x14 dilated,
BiLSTM residuel init 0, BN gelee. Pas de MHA (c'est v20a). Pas de gate T1.

Entree : (B, 6, 24, H, W) grayscale [0, 1]. Ordre des slots :

  0 SAG_FS   1 COR_FS   2 AX_FS
  3 SAG_T1   4 COR_T1   5 AX_T1

Chaque slot est un contraste fixe : fluide (FS) ou T1, sans repli.
Slot absent = volume zero, masque `valid` (comme un plan droppe en v17).
Le T1 n'a plus a battre le fluide dans un 4e slot unique.

Dans train_series.csv, Fluid_Sensitive == Fat_Suppression : pas de fluide
sans FS. AX_T1 remplace le slot vide SAG_FLUID_NOFS du notebook candidat.

LSTM partage sur les 6 sequences (batch = B*S). Epoch 0 recopie v16.
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnet50

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/ResNet50.pt"
MAP_HW = 14
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6


def _dilate_layer4(layer4: nn.Sequential, dilate: int = 2) -> None:
    """stride 32 -> 16, in-place. Poids 3x3 / 1x1 inchanges."""
    for i, block in enumerate(layer4):
        if i == 0:
            block.conv2.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)
        block.conv2.dilation = (dilate, dilate)
        block.conv2.padding = (dilate, dilate)


class _ResNet50Encoder(nn.Module):
    """ResNet-50 jusqu'a layer4 dilated. Cles `backbone.N.*` = v14/v16/v17."""

    def __init__(self) -> None:
        super().__init__()
        try:
            base = resnet50(weights=None)
        except TypeError:
            base = resnet50(pretrained=False)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        _dilate_layer4(self.backbone[7])
        self.num_features = 2048
        for m in self.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class KneeNetV20b(nn.Module):
    arch_id = "kneenet_v20b"

    def __init__(self, backbone: str = "resnet50", n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 16, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.map_hw = MAP_HW
        self.lstm_hidden = int(lstm_hidden)

        self.encoder = _ResNet50Encoder()
        if pretrained:
            self._load_radimagenet(backbone_weights)
        feat_dim = self.encoder.num_features

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()
        self.register_buffer(
            "img_mean", torch.zeros(1, 3, 1, 1), persistent=True,
        )
        self.register_buffer(
            "img_std", torch.ones(1, 3, 1, 1), persistent=True,
        )

    def _init_stem_mix(self) -> None:
        """out_c = in_G pour les 3 canaux -> R=G=B=k au depart."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def _load_radimagenet(self, path: str) -> None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(
                f"poids RadImageNet introuvables : {p.resolve()}"
            )
        sd = torch.load(p, map_location="cpu", weights_only=False)
        self.encoder.load_state_dict(sd, strict=True)
        print(
            f"RadImageNet ResNet-50 charge : {p} | CAM {MAP_HW}x{MAP_HW} "
            f"| BiLSTM hidden {self.lstm_hidden} | slots {self.n_series}",
            flush=True,
        )

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def encoder_parameters(self):
        return self.encoder.parameters()

    def head_parameters(self):
        for name, p in self.named_parameters():
            if not name.startswith("encoder."):
                yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        """rgb : (N, 3, H, W) dans [0, 1] -> (N, C, 14, 14)."""
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            return rgb.new_zeros(0, c, MAP_HW, MAP_HW)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """Triplet A+B pour les tokens valides. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        center = x[bs, ss, dd]
        rgb = torch.stack(
            [
                0.5 + 0.5 * (center - x[bs, ss, d_m]),
                center,
                0.5 + 0.5 * (x[bs, ss, d_p] - center),
            ],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        """gaps (BSD, C) -> residu LSTM (B, S, D, C). Epoch 0 : 0."""
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        """gaps (BSD,C), local (BSD,T,C) -> logits (B,T), slice_attn (B,S,D,T)."""
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )

        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
            gap, loc = self._spatial_pool(maps)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, 14, 14)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw = MAP_HW
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw, hw)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
                spatial = spatial.to(dtype=maps.dtype)
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw, hw,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- Validation du pretraitement : nos volumes DICOM vs stats NPZ ----------
# Au scoring, train_series/ est remplace par test_series/ ; en debug Kaggle,
# train_series/ est disponible : on compare les moments a ceux des NPZ locaux.
import pandas as pd

if DEBUG_COMPARE_NPZ:
    ref_dir = COMP_DIR / 'train_series'
    ref_csv = COMP_DIR / 'train_series.csv'
    if ref_dir.exists() and ref_csv.exists():
        ref = pd.read_csv(ref_csv)
        sample = ref.sample(min(5, len(ref)), random_state=0)
        for _, r in sample.iterrows():
            sdir = ref_dir / r.StudyInstanceUID / r.SeriesInstanceUID
            try:
                v = load_series_dicom(sdir)
                print(f'{r.SeriesInstanceUID[:40]}... shape {v.shape} '
                      f'mean {v.mean():.1f} (NPZ~{NPZ_MEAN_U8}) std {v.std():.1f} (NPZ~{NPZ_STD_U8})')
            except Exception as e:
                print('  [warn]', e)
    else:
        print('train_series/ absent (run de scoring) — comparaison ignoree')


In [ ]:
import pandas as pd

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
           'Contusion', 'Fracture']


def load_v20b(weights_path):
    ckpt = torch.load(str(weights_path), map_location=DEVICE, weights_only=False)
    cfg = ckpt.get('config', {})
    if cfg.get('laterality'):
        raise RuntimeError(f'checkpoint laterality={cfg.get("laterality")!r}, attendu false : {weights_path}')
    if str(cfg.get('model')) != 'kneenet_v20b':
        raise RuntimeError(f"model={cfg.get('model')!r}, attendu kneenet_v20b")
    model = KneeNetV20b(
        backbone=cfg.get('backbone', 'resnet50'),
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=max(8, int(cfg.get('encode_chunk', 8))),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('modele', cfg.get('model'), '| epoch', ckpt.get('epoch'),
          '| swa', ckpt.get('swa_epochs'),
          '| val macro-AUC :', round(float(ckpt.get('val_macro_auc', float('nan'))), 4),
          '| gold_by_epoch :', ckpt.get('gold_by_epoch'),
          '| n_series', model.n_series,
          '| laterality', cfg.get('laterality'))
    return model, cfg


model, cfg = load_v20b(WEIGHTS_PATH)
assert int(model.n_series) == 6, model.n_series
assert LATERALITY_CANON_L is False

# --- Inference : TTA, un seul modele, pretreat = train (pas de flip) ------
import time
import torch.nn.functional as F

test = pd.read_csv(TEST_CSV)
test_series = pd.read_csv(TEST_SERIES_CSV)
print(f'{len(test)} etudes a predire | TTA x{len(TTA_VARIANTS)} | v20b swa solo')


def apply_affine(x, angle_deg=0.0, scale=1.0):
    if angle_deg == 0.0 and scale == 1.0:
        return x
    b, s, d, h, w = x.shape
    xf = x.reshape(b * s * d, 1, h, w)
    angle = x.new_tensor(angle_deg * 3.141592653589793 / 180.0)
    sc = x.new_tensor(scale)
    cos, sin = torch.cos(angle) / sc, torch.sin(angle) / sc
    theta = xf.new_zeros(b, 2, 3)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin
    theta[:, 1, 0] = sin
    theta[:, 1, 1] = cos
    theta = theta.repeat_interleave(s * d, dim=0)
    grid = F.affine_grid(theta, xf.shape, align_corners=False)
    xf = F.grid_sample(xf, grid, align_corners=False, padding_mode='zeros')
    return xf.reshape(b, s, d, h, w)


def _forward(x, use_amp=True):
    global DEVICE, model
    try:
        if use_amp and DEVICE == 'cuda':
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                return model(x)
        return model(x)
    except RuntimeError as e:
        print('forward CUDA echoue, cudnn off puis CPU :', str(e)[:80])
        torch.backends.cudnn.enabled = False
        try:
            if use_amp and DEVICE == 'cuda':
                with torch.autocast('cuda', dtype=AMP_DTYPE):
                    return model(x)
            return model(x)
        except RuntimeError:
            DEVICE = 'cpu'
            model = model.to('cpu')
            return model(x.cpu())


rows = []
t0 = time.time()
n_series = int(model.n_series)
with torch.no_grad():
    for i, uid in enumerate(test.StudyInstanceUID):
        _SERIES_CACHE.clear()
        srows = test_series[test_series.StudyInstanceUID == uid]
        sdir = TEST_SERIES_DIR / uid
        vol = study_to_tensor(sdir, srows, n_series=n_series,
                              laterality=LATERALITY_CANON_L)
        x = torch.from_numpy(vol).unsqueeze(0).to(DEVICE)
        ps = []
        for angle_deg, scale in TTA_VARIANTS:
            xa = apply_affine(x, angle_deg, scale)
            ps.append(torch.sigmoid(_forward(xa, use_amp=True).float()))
        prob = torch.stack(ps, dim=0).mean(dim=0).cpu().numpy()[0]
        rows.append([uid] + prob.tolist())
        if (i + 1) % 50 == 0:
            print(f'{i+1}/{len(test)} | {(time.time()-t0)/60:.1f} min', flush=True)

pred = pd.DataFrame(rows, columns=['StudyInstanceUID'] + TARGETS)
pred.head()


In [ ]:
# --- submission.csv au format exact de sample_submission --------------------
# Obligatoire pour le scoring Kaggle : ecrire /kaggle/working/submission.csv
from pathlib import Path as _Out

sample = pd.read_csv(SAMPLE_SUB)
sub = pred.set_index('StudyInstanceUID').reindex(sample.StudyInstanceUID).reset_index()
sub = sub[list(sample.columns)]
prior = sub[TARGETS].mean()
sub[TARGETS] = sub[TARGETS].fillna(prior)

out_dir = _Out('/kaggle/working') if _Out('/kaggle/working').exists() else _Out('.')
out_csv = out_dir / 'submission.csv'
sub.to_csv(out_csv, index=False)
cwd_csv = _Out('submission.csv')
if cwd_csv.resolve() != out_csv.resolve():
    sub.to_csv(cwd_csv, index=False)

print('ecrit', out_csv.resolve(), '| lignes', len(sub), '| existe', out_csv.exists())
print(sub.shape, '| colonnes OK :', list(sub.columns) == list(sample.columns))
print(sub[TARGETS].describe().loc[['mean', 'std', 'min', 'max']].round(3).to_string())
assert out_csv.exists() and out_csv.stat().st_size > 0, 'submission.csv manquant'
assert not sub[TARGETS].isna().any().any(), 'NaN dans la submission !'
assert list(sub.columns) == list(sample.columns), 'colonnes != sample_submission'
sub.head()
